In [1]:
# BRFSS 2024: Milestone 2 - Data Cleaning & Preprocessing

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, RobustScaler
from scipy import stats

# Load mapped dataset from Milestone 1

df = pd.read_parquet('brfss_2024_mapped_vars.parquet')
print("Mapped dataset loaded. Shape:", df.shape)

# Identify numeric health variables for preprocessing

health_patterns = ['BMI', 'ASTH', 'MICHD', 'RFHLTH', 'PHYS14D', 'MENT14D', 'HLTHPL2', 'DENVST3', 'DIABETES', 'HYPERTENSION', 'CHOLESTEROL']
health_vars = [c for c in df.columns if any(p.lower() in c.lower() for p in health_patterns)]
print("Health variables for preprocessing:", health_vars)

# M2.T1: Handle missing values

# Flag features with >20% missing
missing_percent = df[health_vars].isnull().mean() * 100
high_missing = missing_percent[missing_percent > 20]
print("Features with >20% missing:\n", high_missing)

# Median imputation for numeric variables
numeric_imputer = SimpleImputer(strategy='median')
numeric_cols = df[health_vars].select_dtypes(include=np.number).columns.tolist()
df[numeric_cols] = numeric_imputer.fit_transform(df[numeric_cols])

# M2.T2: Encode ordinal variables (if any)
ordinal_vars = [c for c in health_vars if 'ASTH' in c]  # example: asthma severity
if ordinal_vars:
    ordinal_encoder = OrdinalEncoder()
    df[ordinal_vars] = ordinal_encoder.fit_transform(df[ordinal_vars])
    print("Ordinal variables encoded:", ordinal_vars)

# M2.T3: Scale numeric features
robust_scaler = RobustScaler()
df[numeric_cols] = robust_scaler.fit_transform(df[numeric_cols])
print("Numeric variables scaled using RobustScaler.")

# M2.T4: Outlier detection using z-score
z_scores = np.abs(stats.zscore(df[numeric_cols]))
outliers = (z_scores > 3).any(axis=1)
df['outlier_flag'] = outliers
print(f"Number of outliers detected: {outliers.sum()}")

# Flagged outliers for review
df_clean = df.copy()

# Save cleaned dataset for Milestone 3
df_clean.to_parquet('brfss_2024_cleaned_qusipe.parquet', index=False)
print("Cleaned dataset saved as 'brfss_2024_cleaned_qusipe.parquet'")

Mapped dataset loaded. Shape: (409415, 46)
Health variables for preprocessing: ['ASTHMA3', 'ASTHNOW', 'BMI', 'BMI_raw', 'CASTHDX2', 'CASTHNO2', '_ASTHMS1', '_BMI5CAT', '_CASTHM1', '_DENVST3', '_HLTHPL2', '_LTASTH1', '_MENT14D', '_MICHD', '_PHYS14D', '_RFBMI5', '_RFHLTH']
Features with >20% missing:
 ASTHNOW     84.335943
CASTHDX2    89.134741
CASTHNO2    98.909175
dtype: float64
Ordinal variables encoded: ['ASTHMA3', 'ASTHNOW', 'CASTHDX2', 'CASTHNO2', '_ASTHMS1', '_CASTHM1', '_LTASTH1']
Numeric variables scaled using RobustScaler.
Number of outliers detected: 92043
Cleaned dataset saved as 'brfss_2024_cleaned_qusipe.parquet'
